# Week 2: Pandas Data Wrangling
## Notebook 06: Advanced DateTime Manipulation & Window Functions from Scratch

### 1. Conceptual Intuition & Real-World Use Case
In Financial Trading, IoT Sensor Monitoring, and Demand Forecasting:
- Single-point observations (e.g. "Stock price today is $150") carry little predictive power on their own.
- Predictive signal lies in **temporal dynamics**: Moving averages, rolling volatility, exponential momentum, and lag/lead indicators.

Pandas provides industry-standard temporal primitives:
1. **Rolling Windows:** Fixed-interval statistics (e.g., 7-day Simple Moving Average).
2. **Exponentially Weighted Moving Average (EWMA):** Weights recent observations exponentially higher than older ones.
3. **Lag / Lead Shifts:** Comparing current values against past time steps without lookahead leakage.

---

### 2. Mathematical Foundations: Simple vs. Exponential Moving Averages
1. **Simple Moving Average (SMA with window $W$):**
   $$\text{SMA}_t = \frac{1}{W} \sum_{i=0}^{W-1} x_{t-i}$$

2. **Exponentially Weighted Moving Average (EWMA):**
   $$y_t = (1 - \alpha) y_{t-1} + \alpha x_t$$
   Where smoothing factor $\alpha \in (0, 1]$ can be parameterized by half-life $H$:
   $$\alpha = 1 - \exp\left( \frac{-\ln 2}{H} \right)$$
   Unlike SMA which drops older points abruptly (causing artificial jumps), EWMA decays smoothly over time.

In [ ]:
import numpy as np
import pandas as pd

# Step 1: Create Hourly Sensor / Stock Price Data for 10 Days
dates = pd.date_range(start="2026-10-01", periods=100, freq="h")
np.random.seed(42)

# Generate Random Walk with Trend
returns = np.random.normal(loc=0.002, scale=0.02, size=len(dates))
price_series = 100 * np.cumprod(1 + returns)

df = pd.DataFrame({
    "timestamp": dates,
    "price": price_series
}).set_index("timestamp")

print("Raw Time-Series Data (First 5 rows):")
print(df.head())

# Step 2: Feature Engineering with Rolling Windows
# 1. 24-Hour Rolling Simple Moving Average (SMA)
df["sma_24h"] = df["price"].rolling(window="24h").mean()

# 2. 24-Hour Rolling Volatility (Standard Deviation)
df["volatility_24h"] = df["price"].rolling(window="24h").std()

# 3. Exponentially Weighted Moving Average (Half-life = 12 hours)
df["ewma_12h"] = df["price"].ewm(halflife="12h", times=df.index).mean()

# Step 3: Lag and Lead (Shift) Features
# Lag 1: Price 1 hour ago (Past feature - safe for ML training)
df["lag_1h"] = df["price"].shift(1)

# Lag 24: Price 24 hours ago (Daily seasonality)
df["lag_24h"] = df["price"].shift(24)

# Percentage Return over past 24 hours
df["return_24h"] = (df["price"] - df["lag_24h"]) / df["lag_24h"]

# Target: Next hour's price return (Lead 1 - Target variable ONLY)
df["target_next_return"] = (df["price"].shift(-1) - df["price"]) / df["price"]

print("\n--- Feature-Engineered Time-Series (Rows 25 to 29) ---")
print(df[["price", "sma_24h", "ewma_12h", "return_24h", "target_next_return"]].iloc[25:30])

### 3. Step-by-Step Code Walkthrough
1. **Datetime Indexing:** Using `set_index('timestamp')` enables time-aware offset strings like `'24h'` inside rolling calculations.
2. **Rolling Volatility:** Measures price turbulence over the trailing 24 hours.
3. **Lag vs Lead:** `shift(1)` looks backward (preventing data leakage), while `shift(-1)` creates the future target variable for supervised forecasting models.

---

### 4. Industry Applications & Trade-offs
- **High-Frequency Trading:** Calculating MACD (Moving Average Convergence Divergence) and Bollinger Bands.
- **Smart Energy Grids:** Predicting peak kilowatt demand using 24-hour and 7-day seasonal lags.

---

### 5. Top Interview Questions & Answers
**Q1: What is the primary cause of lookahead bias when creating time-series features?**
> **Answer:** Lookahead bias occurs when a feature uses future information that was not available at the timestamp of prediction. For example, centering a rolling window (`center=True`) calculates an average using both past and future data points, causing models to appear artificially accurate in backtests while failing in production.

**Q2: What is the difference between SMA and EWMA?**
> **Answer:** SMA gives equal weight to all $W$ observations in the window and completely drops observations older than $W$, creating discrete shock artifacts when extreme values drop out. EWMA assigns exponentially decaying weights, making it smoother and more reactive to recent shocks.